In [ ]:
#!pip install optuna shap imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm

from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, lit, isnan, count, mean, date_format
from pyspark.ml.feature import Imputer
from pyspark.sql.types import DateType
from pyspark.ml.feature import VectorAssembler, StandardScaler

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, brier_score_loss, average_precision_score, make_scorer
import optuna
import shap
from scipy.stats import ks_2samp
from imblearn.under_sampling import RandomUnderSampler
%md
#Funções pré definidas
def atribuir_faixa_score(score, quantiles):
  """
  Função para atribuir 10 faixas de score, de A a J, com base em uma lista de quantiles,
  indicando também o intervalo no formato (xxx ; yyy].
  Exemplo: df['faixa_score'] = df['score'].apply(lambda x: atribuir_faixa_score(x, quantiles))
  """
  if score <= quantiles[0]:
    return 'A: (0 ; ' + str(np.round(quantiles[0], 0)) + ']'
  elif ((score > quantiles[0]) & (score <= quantiles[1])):
    return 'B: (' + str(np.round(quantiles[0], 0)) + ' ; ' + str(np.round(quantiles[1], 0)) + ']'
  elif ((score > quantiles[1]) & (score <= quantiles[2])):
    return 'C: (' + str(np.round(quantiles[1], 0)) + ' ; ' + str(np.round(quantiles[2], 0)) + ']'
  elif ((score > quantiles[2]) & (score <= quantiles[3])):
    return 'D: (' + str(np.round(quantiles[2], 0)) + ' ; ' + str(np.round(quantiles[3], 0)) + ']'
  elif ((score > quantiles[3]) & (score <= quantiles[4])):
    return 'E: (' + str(np.round(quantiles[3], 0)) + ' ; ' + str(np.round(quantiles[4], 0)) + ']'
  elif ((score > quantiles[4]) & (score <= quantiles[5])):
    return 'F: (' + str(np.round(quantiles[4], 0)) + ' ; ' + str(np.round(quantiles[5], 0)) + ']'
  elif ((score > quantiles[5]) & (score <= quantiles[6])):
    return 'G: (' + str(np.round(quantiles[5], 0)) + ' ; ' + str(np.round(quantiles[6], 0)) + ']'
  elif ((score > quantiles[6]) & (score <= quantiles[7])):
    return 'H: (' + str(np.round(quantiles[6], 0)) + ' ; ' + str(np.round(quantiles[7], 0)) + ']'
  elif ((score > quantiles[7]) & (score <= quantiles[8])):
    return 'I: (' + str(np.round(quantiles[7], 0)) + ' ; ' + str(np.round(quantiles[8], 0)) + ']'
  else:
    return 'J: (' + str(np.round(quantiles[8], 0)) + ' ; ' + '1000]'

def faixas_score(df, quantiles):
  """
  Função que realiza a contagem do volume e da target por faixa de score, indicando também,
  os respectivos valores percentuais.
  Utilizada para ver o resultado geral, considerando todas as safras presentes na amostra.
  Exemplo: fxs_train = faixas_score(df_fx_train, quantiles)
  """
  df['faixa_score'] = df['score'].apply(lambda x: atribuir_faixa_score(x, quantiles))
  # Agrupar por faixa de score e calcular as métricas desejadas
  faixa_score_stats = df.groupby('faixa_score').agg(
  volume=('target', 'size'), # Total de observações (volume)
  target=('target', 'sum'), # Total de y_true = 1 (target)
  )
  # Calcular o percentual do volume de observações
  faixa_score_stats['percentual_volume'] = (faixa_score_stats['volume'] / len(df)) * 100
  # Calcular o percentual de y_true = 1
  faixa_score_stats['percentual_target'] = (faixa_score_stats['target'] / faixa_score_stats['volume']) * 100
  return faixa_score_stats

def faixas_score_safras(df, quantiles):
  """
  Semelhante a função faixas_score(df, quantiles). Realiza a contagem do volume, target e
  cálculo dos respectivos percentuais, considerando cada safra.
  Cada dataframe por safra é armazenado em uma lista.
  Exemplo: DF, safras = faixas_score_safras(df_fx_train, quantiles)
  """
  safras = df['Safra'].unique().tolist()
  DF = []
  for s in safras:
    df_aux = df[df['Safra'] == s].copy()
    DF.append(faixas_score(df_aux, quantiles))
  return DF, safras

################################################################################

def ks2_total(y_true, y_pred):
  """
  Cálculo do KS2 total, considerando todas as safras.
  Exemplo: ks_treino = ks2_total(y_train,y_pred_train)
  """
  ks2_tot = ks_2samp(y_pred[y_true == 1], y_pred[y_true == 0]).statistic
  return ks2_tot

def ks_safra(df):
  """
  Cálculo do KS2 por safra.
  Retorna um dataframe com as safras e os respectivos KS2.
  Exemplo: df_ks = ks_safra(df)
  """
  # Calcular KS Score por safra
  ks_results = {}
  for safra in df["Safra"].unique():
    df_safra = df[df["Safra"] == safra].copy()
    # Separar scores preditos por classe
    scores_positivos = df_safra[df_safra["target"] == 1]["probs"]
    scores_negativos = df_safra[df_safra["target"] == 0]["probs"]
    # Aplicar KS Test
    ks_stat, _ = ks_2samp(scores_positivos, scores_negativos)
    # Armazenar resultado
    ks_results[safra] = ks_stat

  # Exibir os resultados
  ks_df = pd.DataFrame(list(ks_results.items()), columns=["Safra", "KS_Score"])
  return(ks_df)
def plot_ks2_por_safra(df_ks, amostra):
  """
  Função que plota o gráfico de linha do KS2 por safra.
  """
  plt.figure(figsize=(12, 4))
  plt.plot(df_ks["Safra"], df_ks["KS_Score"], marker='o', linestyle='dashed', color='blue')
  plt.ylim(0, 1)
  plt.xlabel("Safras")
  plt.ylabel("KS Score")
  plt.title("KS2 por safra - "+ amostra)
  # Adicionar valores numéricos em cima dos pontos
  x = np.array(df_ks["Safra"].to_list())
  y = np.round(np.array(df_ks["KS_Score"].to_list()), 2)
  for i, j in zip(x,y):
    plt.text(i, j+0.05, str(j), ha='center', fontsize=10)

  plt.show()

def plot_tx_target_total(df, amostra):
  """
  Função que plota o gráfico de linha da taxa target com gráfico de barras do percentual volume.
  """
  fig, ax1 = plt.subplots(figsize=(12, 4))
  # Criando o gráfico de linhas no eixo primário
  ax1.plot(df['faixa_score'], df['percentual_target'], color='red', marker='o', linestyle='--', linewidth=2, label='Target')
  ax1.set_xlabel('Faixas de score')
  ax1.set_ylabel('(%) Target')
  ax1.set_xticks(range(len(df['faixa_score'])))
  ax1.set_xticklabels(ax1.get_xticklabels(), rotation=60)
  ax1.set_ylim(0, df['percentual_target'].max()+10)
  ax1.tick_params(axis='y')

  x = np.array(df['faixa_score'].to_list())
  y = np.round(np.array(df['percentual_target'].to_list()), 3)
  for i, j in zip(x,y):
    ax1.text(i, j+1.5, str(j), ha='left', fontsize=10)

  # Criando um segundo eixo Y
  ax2 = ax1.twinx()
  ax2.bar(df['faixa_score'], df['percentual_volume'], color='blue', alpha=0.2, label='Volume')
  ax2.set_ylabel('(%) Volume')
  ax2.set_ylim(0, df['percentual_volume'].max()+5)
  ax2.tick_params(axis='y')
  # Adicionando legendas
  ax1.legend(loc='upper left')
  ax2.legend(loc='upper right')
  plt.title("Taxa target total por faixa - "+ amostra)
  # # Adicionar valores numéricos em cima dos pontos

  plt.show()

def plot_tx_target_por_faixa(df, amostra, fx_labels):
    """
    Função que plota o gráfico de linhas da taxa target por faixas de score e safras.
    """
    # Escolher um colormap divergente: coolwarm, bwr, seismic
    colormap = cm.seismic
    # Selecionar 10 pontos distribuídos no colormap
    num_cores = 10
    indices = np.linspace(0, 1, num_cores) # De 0 (cor inicial) a 1 (cor final)
    cores_custom = colormap(indices)

    fig, axs = plt.subplots(1, 2, figsize=(16, 4))

    # Primeiro subplot para faixas A, B e C
    axs[0].plot(df["Safra"], df['A'], label=fx_labels[0], color=cores_custom[9])
    axs[0].plot(df["Safra"], df['B'], label=fx_labels[1], color=cores_custom[8])
    axs[0].plot(df["Safra"], df['C'], label=fx_labels[2], color=cores_custom[7])
    axs[0].plot(df["Safra"], df['D'], label=fx_labels[3], color=cores_custom[6])
    axs[0].plot(df["Safra"], df['E'], label=fx_labels[4], color=cores_custom[5])
    axs[0].plot(df["Safra"], df['F'], label=fx_labels[5], color=cores_custom[4])
    axs[0].plot(df["Safra"], df['G'], label=fx_labels[6], color=cores_custom[3])
    axs[0].plot(df["Safra"], df['H'], label=fx_labels[7], color=cores_custom[2])
    axs[0].plot(df["Safra"], df['I'], label=fx_labels[8], color=cores_custom[1])
    axs[0].plot(df["Safra"], df['J'], label=fx_labels[9], color=cores_custom[0])
    #axs[0].set_ylim(0, df['A'].max()+5)
    axs[0].set_xlabel("Safras")
    axs[0].set_ylabel("(%) Target")
    axs[0].set_title("ever90mob12 - " + amostra)
    axs[0].set_xticks(np.arange(len(df["Safra"].unique())))
    axs[0].set_xticklabels(df["Safra"], rotation=60)
    axs[0].legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.65))

    # Segundo subplot para faixas D, E, F, G, H, I, J
    # axs[1].plot(df["Safra"], df['A'], label=fx_labels[0], color=cores_custom[9])
    # axs[1].plot(df["Safra"], df['B'], label=fx_labels[1], color=cores_custom[8])
    # axs[1].plot(df["Safra"], df['C'], label=fx_labels[2], color=cores_custom[7])
    # axs[1].plot(df["Safra"], df['D'], label=fx_labels[3], color=cores_custom[6])
    # axs[1].plot(df["Safra"], df['E'], label=fx_labels[4], color=cores_custom[5])
    axs[1].plot(df["Safra"], df['F'], label=fx_labels[5], color=cores_custom[4])
    axs[1].plot(df["Safra"], df['G'], label=fx_labels[6], color=cores_custom[3])
    axs[1].plot(df["Safra"], df['H'], label=fx_labels[7], color=cores_custom[2])
    axs[1].plot(df["Safra"], df['I'], label=fx_labels[8], color=cores_custom[1])
    axs[1].plot(df["Safra"], df['J'], label=fx_labels[9], color=cores_custom[0])
    #axs[1].set_ylim(0, df['E'].max()+df['E'].mean())
    axs[1].set_xlabel("Safras")
    axs[1].set_ylabel("(%) Target")
    axs[1].set_title("ever90mob12 (zoom) - " + amostra)
    axs[1].set_xticks(np.arange(len(df["Safra"].unique())))
    axs[1].set_xticklabels(df["Safra"], rotation=60)
    axs[1].legend(ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.6))

    #plt.tight_layout()
    plt.show()

def plot_tx_volume_por_faixa(df, amostra,fx_labels):
  """
  Função que plota o gráfico de barras empilhadas indicando os percentuais volume dentro de cada faixa por safra.
  """
  # Escolher um colormap divergente: coolwarm, bwr, seismic
  colormap = cm.seismic
  # Selecionar 10 pontos distribuídos no colormap
  num_cores = 10
  indices = np.linspace(0, 1, num_cores) # De 0 (cor inicial) a 1 (cor final)
  cores_custom = colormap(indices)
  # Criando o gráfico de barras empilhadas
  fig, ax = plt.subplots(figsize=(12, 4))
  ax.bar(df["Safra"], df['A'], label=fx_labels[0], color=cores_custom[9])
  ax.bar(df["Safra"], df['B'], label=fx_labels[1], bottom=df['A'], color=cores_custom[8])
  ax.bar(df["Safra"], df['C'], label=fx_labels[2], bottom=df['A'] + df['B'], color=cores_custom[7])
  ax.bar(df["Safra"], df['D'], label=fx_labels[3], bottom=df['A'] + df['B'] + df['C'], color=cores_custom[6])
  ax.bar(df["Safra"], df['E'], label=fx_labels[4], bottom=df['A'] + df['B'] + df['C'] + df['D'], color=cores_custom[5])
  ax.bar(df["Safra"], df['F'], label=fx_labels[5], bottom=df['A'] + df['B'] + df['C'] + df['D'] + df['E'], color=cores_custom[4])
  ax.bar(df["Safra"], df['G'], label=fx_labels[6], bottom=df['A'] + df['B'] + df['C'] + df['D'] + df['E'] + df['F'], color=cores_custom[3])
  ax.bar(df["Safra"], df['H'], label=fx_labels[7], bottom=df['A'] + df['B'] + df['C'] + df['D'] + df['E'] + df['F'] + df['G'], color=cores_custom[2])
  ax.bar(df["Safra"], df['I'], label=fx_labels[8], bottom=df['A'] + df['B'] + df['C'] + df['D'] + df['E'] + df['F'] + df['G'] + df['H'], color=cores_custom[1])
  ax.bar(df["Safra"], df['J'], label=fx_labels[9], bottom=df['A'] + df['B'] + df['C'] + df['D'] + df['E'] + df['F'] + df['G'] + df['H'] + df['I'], color=cores_custom[0])
  # Adicionando rótulos e título
  ax.set_xlabel('Safras')
  ax.set_ylabel('Percentual volume')
  ax.set_ylim(0, 105)
  ax.set_title('Percentual volume por faixa - '+ amostra) #amostra

  ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.35))
  # Exibir o gráfico
  plt.show()

def graficos_analise_estabilidade(df, quantiles, amostra):
  """
  Função que faz a plotagem de todos os gráficos necessários para a análise da estabilidade.
  Exemplo: graficos_analise_estabilidade(df_fx_train, quantiles, amostra='Base Treino')
  """
  DF, safras = faixas_score_safras(df, quantiles)
  # Gráfico de linha do KS2 por safra
  print("KS2 por safra:")
  df_ks = ks_safra(df)
  # plota gráfico
  plot_ks2_por_safra(df_ks, amostra)
  # Gráfico de linha da taxa target com gráfico de barras do percentual volume
  print("\nPercentual da target na base:")
  df_tx_target = faixas_score(df, quantiles)[['percentual_target', 'percentual_volume']].copy().reset_index(drop=False)
  faixa_labels = df_tx_target['faixa_score'].to_list()
  # plota gráfico
  plot_tx_target_total(df_tx_target, amostra)

  # Gráfico de linhas da taxa target por faixas de score e safras
  print("\nPercentual da target por faixa:")
  target = []
  for i in range(len(safras)):
    target.append(DF[i]['percentual_target'].to_list())

  df_target = pd.DataFrame(target, columns=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'])
  df_target['Safra'] = safras
  df_target["Safra"] = df_target["Safra"].astype(object)
  # plota gráfico
  plot_tx_target_por_faixa(df_target, amostra, faixa_labels)
  # Gráfico de barras empilhadas indicando os percentuais volume dentro de cada faixa por safra
  print("\nPercentual da volumetria por faixa:")
  volume = []
  for i in range(len(safras)):
    volume.append(DF[i]['percentual_volume'].to_list())

  df_volume = pd.DataFrame(volume, columns=['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'])
  df_volume['Safra'] = safras
  df_volume["Safra"] = df_volume["Safra"].astype(object)

  # plota gráfico
  plot_tx_volume_por_faixa(df_volume, amostra, faixa_labels)
%md
# Importa base
#Importa base
df = spark.createDataFrame(data_science.df_dummy_model_activation_FE)
# Separar OOT e dividir em 70% treinamento e 30% teste, com semente fixa para reprodutibilidade
df = df.withColumn("Safra", date_format(col("dat_abertura_conta"), "yyyy-MM"))
recentes_df = df.filter(col("Safra") >= "2024-07")
oot_df = df.filter(col("Safra") >= "2023-07" & col("Safra") < "2024-07")
train_df, test_df = df.filter(col("Safra") < "2023-07").randomSplit([0.7, 0.3], seed=42)

print(train_df.count())
print(test_df.count())
print(oot_df.count())
%md
#Feature Enginneer
%md
### 1. One Hot Encoding (OHE)
train_df.dtypes
#função para transformar variáveis categóricas em flags
categorical_cols = ['cod_estado_civil','nom_estado','cod_canal_venda','cod_logo']
index_output_cols = [x + "Index" for x in categorical_cols]
ohe_output_cols = [x + "OHE" for x in categorical_cols]

string_indexer = StringIndexer(inputCols=categorical_cols, outputCols=index_output_cols, handleInvalid="skip")
string_indexer_fit = string_indexer.fit(df_train)
df_train_string_index = string_indexer_fit.transform(df_train)

ohe_encoder = OneHotEncoder(inputCols=index_output_cols, outputCols=ohe_output_cols)
ohe_encoder_fit = ohe_encoder.fit(df_train_string_index)
df_train_ohe = ohe_encoder_fit.transform(df_train_string_index)
%md
##2. Tratamento de nulos
numerical_cols = ['vlr_limite_credito_inicial','renda_risco','hspn_venda','qtd_dependente','idade','limite_inicial/renda_risco','limite_inicial/salario','renda_risco/salario']

imputer = Imputer(inputCols=numerical_cols, outputCols=[col + "_imputed" for col in numerical_cols])
imputer_fit = imputer.fit(df_train_ohe)
df_train_imputer = imputer_fit.transform(df_train_ohe)
%md
## 3. Normalização
# Primeiro monta o vetor de features numéricas imputadas
assembler = VectorAssembler(inputCols=[col + "_imputed" for col in numerical_cols],
                            outputCol="numerical_features_vec")
df_train_vector = assembler.transform(df_train_imputer)

# Então normaliza
scaler = StandardScaler(inputCol=assembled_input, outputCol="numerical_features_scaled", withMean=True, withStd=True)
scaler_fit = scaler.fit(df_publico_target_spark_f)
df_train_scaler = scaler_fit.transform(df_train_vector)
%md
#Feature Selection
%md
### 1. Via modelo
Cria um modelo simples para selecionar variáveis mais importantes
X_train = df_train_scaler.drop("Safra", "target").toPandas()  # Features
y_train = df_train_scaler.select('target').toPandas().values.ravel().astype(int) # Target
# Cria e treina um modelo de classificação LightGBM com seed para replicabilidade
fs_model = lgb.LGBMClassifier(random_state=42, verbosity=-1)
fs_model.fit(X_train_amostra, y_train_amostra)

# Coleta a importância das features atribuída pelo modelo treinado
importances = fs_model.feature_importances_
feature_names = fs_model.feature_name_

# Cria um DataFrame com os nomes e importâncias das features
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Ordena pela importância, do maior para o menor
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Calcula a importância acumulada
total_importance = feature_importance_df['Importance'].sum()
cumulative_importance = feature_importance_df['Importance'].cumsum()
feature_importance_df['Cumulative'] = cumulative_importance / total_importance

# Seleciona apenas as features que juntas somam até 90% da importância acumulada
selected_features = feature_importance_df[feature_importance_df['Cumulative'] <= 0.90]['Feature'].tolist()

# Imprime a lista final de features selecionadas
print(selected_features)
%md
###2. Via correlação
Entre as mais importante, limpa correlação alta (por exemplo, acima de 0.6)
# Selecionar apenas features numéricas
X_train_num = X_train[selected_features].select_dtypes(include=['float64'])

# Calcular a matriz de correlação
correlation_matrix = X_train_num.corr()

# Gerar o correlograma
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlograma das variáveis numéricas mais imporantes')
plt.show()
#Listar somente as variáveis que vão ser testadas no modelo - comentar altamente correlacionadas
selected_features_corr = ['MEAN_VLRMULTJURPAGO',
 'MEAN_VLRATRASO',
 'vlr_restritivo',
 'qtd_meses_tmp_prim_relac_bcp_bcp',
 #'vlr_renda_risco',
 'qtd_idade_anos',
 'MEAN_PERCTXADMPAGA',
 'MEAN_PERCFDOAQUISICAO',
 'qtd_dias_entrega_bem',
 'flg_cotas_emdia',
 'qtd_meses_restantes',
 'MEAN_VLRLANCNORMPAGO',
 'MEAN_vlr_diff_cart_cred',
 'MEAN_VLRLANCEMBPAGO',
 'flg_possui_aut',
 'flg_possui_imv',
 'flg_grupo_andamento',
 'qtd_total_cotas',
 'flg_norte_nordeste',
#  'MEAN_PERC_REDUTOR',
 #'flg_plano_redutor',
 'vlr_parcela_sob_renda']
%md

%md
#Model
X_train_model = X_train[selected_features]
%md
###1. Amostragem
#Undersample balanceado 50%
undersampler = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_train_resampled, y_train_resampled = undersampler.fit_resample(X_train_amostra, y_train_amostra)
y_train_resampled = pd.DataFrame(y_train_resampled, columns=['target'])
print(y_train.value_counts(normalize=True))
print(y_train_resampled['target'].value_counts(normalize=True))
%md
###2. Otimização de hiperparâmetros
#Define a função objetivo que será otimizada

def objective(trial):
    # Define o espaço de busca para os hiperparâmetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }

    # Cria o modelo com os parâmetros sugeridos
    model = lgb.LGBMClassifier(**params)

    # Validação cruzada estratificada para avaliar AUC
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc = cross_val_score(model, X_train_resampled, y_train_resampled,
                          scoring=make_scorer(roc_auc_score, needs_proba=True),
                          cv=cv).mean()

    # Retorna a métrica que será maximizada
    return auc

#Configura o estudo
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(sampler=sampler, direction='maximize')  # AUC deve ser maximizada
#Roda otimização
study.optimize(objective, n_trials=15)  # 15 tentativas de combinações de hiperparâmetros, escolhidas de forma otimizada
#Melhores resultados:
print("Melhores hiperparâmetros:", study.best_trial.params)
# Visualização Fronteira de Pareto caso use múltiplas métricas
optuna.visualization.plot_pareto_front(study).show()

# Importância dos hiperparâmetros
optuna.visualization.plot_param_importances(study).show()
%md
###3. Treinamento
#Declara os melhores parâmetros encontrados pelo optuna
params={'n_estimators': 100, 'learning_rate': 0.09778750519348421, 'num_leaves': 17, 'max_depth': 9, 'feature_fraction': 0.5024892755136359, 'feature_fraction_bynode': 0.6908844309006715, 'min_child_weight': 0.2584541291654643, 'max_delta_step': 4.58793077046451, 'min_data_in_leaf': 79, 'min_child_samples': 41, 'subsample': 0.5008940682011966, 'colsample_bytree': 0.7882288650456512, 'reg_alpha': 0.11697053640329573, 'reg_lambda': 2.389932286461989e-09, 'max_bin': 28}

# Criar o modelo final com os melhores hiperparâmetros
final_model = lgb.LGBMClassifier(**params,random_state=42,verbose = -1,boosting = "dart")

# Treinar o modelo final
y_train_resampled = y_train_resampled.values.ravel().astype(int)
final_model.fit(X_train_resampled, y_train_resampled)
%md
###4. Avaliação do modelo
# Resultado base de treinamento amostrada
y_pred_train_resampled = final_model.predict_proba(X_train_resampled)[:, 1]

# Resultado base de treinamento original
y_pred_train = final_model.predict_proba(X_train)[:, 1]

# Resultado base de teste (na verdade aqui ainda precisa aplicar todos os tratamentos na base de teste)
X_oot = test_df[selected_features].copy()
y_pred_test = final_model.predict_proba(X_test)[:, 1]

# Resultado base OOT (na verdade aqui ainda precisa aplicar todos os tratamentos na base OOT)
X_oot = oot_df[selected_features].copy()
y_pred_oot = final_model.predict_proba(X_oot)[:, 1]

# Resultado base safras recentes (na verdade aqui ainda precisa aplicar todos os tratamentos na base de safras recente)
X_recentes = recentes_df[selected_features].copy()
y_pred_recentes = final_model.predict_proba(X_recentes)[:, 1]
%md
#### 4.1 KS
print("Treino amostrado - KS:", ks2_total(y_train_resampled, y_pred_train_resampled))
print("Treino original - KS:", ks2_total(y_train, y_pred_train))
print("Teste - KS:", ks2_total(y_test, y_pred_test))
print("OOT - KS:", ks2_total(y_oot, y_pred_oot))
#Gráfico para comparação

ks_treino = ks2_total(y_train,y_pred_train)
ks_oot = ks2_total(y_oot,y_pred_oot)
ks_diff = ks_treino - ks_oot

# Dados para o gráfico
ks_values = [ks_treino, ks_oot, ks_diff]
labels = ['ks_treino', 'ks_oot', 'ks_diff']
# Criar o gráfico de barras
plt.figure(figsize=(8, 5))
bars = plt.bar(labels, ks_values, color=['blue', 'red', 'gray'])
# Adicionar os valores numéricos em cima de cada barra
for bar, value in zip(bars, ks_values):
  plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{value:.4f}', ha='center', va='bottom')

# Configurar os rótulos e título do gráfico
plt.xlabel('Métricas KS')
plt.ylabel('Valores')
plt.title('Comparação das Métricas KS')

#ks_summary_path = "/dbfs/tmp/ks_bhv_con_pf_01_summary.png"
#plt.savefig(ks_summary_path)

# Exibir o gráfico
plt.show()
%md
#### 4.2 AUC
print("Treino amostrado - AUC:", roc_auc_score(y_train_resampled, y_pred_train_resampled))
print("Treino original - AUC:", roc_auc_score(y_train, y_pred_train))
print("Teste - AUC:", roc_auc_score(y_test, y_pred_test))
print("OOT - AUC:", roc_auc_score(y_oot, y_pred_oot))
%md
#### 4.3 Ordenação
def calcular_decis(df, score_col='score', target_col='target', bins=10, score_bins=None):
    """
    Quebra uma base em decis com base na coluna de score, e calcula a taxa de target por decil.
    Se score_bins for passado, aplica a quebra já definida (útil para manter corte igual entre bases).
    """
    df = df.copy()

    # Se não passou os bins, calcula com base nos quantis do score
    if score_bins is None:
        score_bins = pd.qcut(df[score_col], q=bins, retbins=True, duplicates='drop')[1]

    # Aplica a quebra com os bins
    df['decil'] = pd.cut(df[score_col], bins=score_bins, labels=[f'D{10 - i}' for i in range(bins)], include_lowest=True)

    # Calcula a taxa de target
    result = df.groupby('decil')[target_col].agg(['count', 'sum'])
    result['taxa_target_%'] = (result['sum'] / result['count']) * 100
    result = result.rename(columns={'count': 'volume', 'sum': 'target_1'})

    return result.reset_index(), score_bins

# Supondo que você já tenha X_train com coluna 'score' e 'target' adicionada

df_train_eval = X_train.copy()
df_train_eval['score'] = y_pred_train
df_train_eval['target'] = y_train

df_test_eval = X_test.copy()
df_test_eval['score'] = y_pred_test
df_test_eval['target'] = y_test

df_oot_eval = X_oot.copy()
df_oot_eval['score'] = y_pred_oot
df_oot_eval['target'] = y_oot

# 1. Quebrar treino em decis e pegar os bins
decis_train, bins = calcular_decis(df_train_eval)

# 2. Usar os mesmos bins para teste e oot
decis_test, _ = calcular_decis(df_test_eval, score_bins=bins)
decis_oot, _ = calcular_decis(df_oot_eval, score_bins=bins)

# Visualizar resultados
print("Treino")
display(decis_train)
print("Teste")
display(decis_test)
print("OOT")
display(decis_oot)
%md
####4.4 Feature Importante e SHAP
# Plot feature importance
lgb.plot_importance(final_model, figsize=(10, 6))
plt.title("Feature Importance")
plt.show()

#feature_importance_summary_path = "/dbfs/tmp/feature_importance_bhv_con_pf_01_summary.png"
#plt.savefig(feature_importance_summary_path)
# Plot feature importance - curva acumulada
importances = final_model.feature_importances_
feature_names = final_model.feature_name_
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Ordenar pela importância
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
total_importance = feature_importance_df['Importance'].sum()
cumulative_importance = feature_importance_df['Importance'].cumsum()

feature_importance_df['Importancia'] = (feature_importance_df['Importance'] / total_importance)*100
feature_importance_df['Cumulative'] = (cumulative_importance / total_importance)*100

feature_importance_df = feature_importance_df.round(2)
feature_importance_df = feature_importance_df.drop(columns=['Importance'])
feature_importance_df

# plt.figure(figsize=(12, 6))
# plt.plot(feature_importance_df['Feature'], feature_importance_df['Cumulative'], marker='o')
# plt.xlabel('Features')
# plt.ylabel('Importância Acumulada')
# plt.title('Gráfico de Importância Acumulada das Features')
# plt.xticks(rotation=80)
# plt.ylim(0, 1.05)
# plt.tight_layout()
# plt.grid(True, axis='y')
# plt.show()
#SHAP
explainer = shap.Explainer(final_model)
shap_values = explainer(X_train)
shap.summary_plot(shap_values[:,:,1], X_train, max_display=25, show=False)
#shap_summary_path = "/dbfs/tmp/shap_bhv_con_pf_01_summary.png"
#plt.savefig(shap_summary_path)
%md
###5. Estabilidade
%md
**KS por safra**
from scipy.stats import ks_2samp

def calcular_ks_por_safra(df, score_col='score', target_col='target', safra_col='Safra'):
    """
    Calcula o KS por safra usando a coluna de score e target.
    """
    ks_resultados = []

    for safra in sorted(df[safra_col].unique()):
        df_safra = df[df[safra_col] == safra]
        scores_positivos = df_safra[df_safra[target_col] == 1][score_col]
        scores_negativos = df_safra[df_safra[target_col] == 0][score_col]

        ks = ks_2samp(scores_positivos, scores_negativos).statistic
        ks_resultados.append((safra, ks))

    ks_df = pd.DataFrame(ks_resultados, columns=['Safra', 'KS_Score'])
    return ks_df

# Adiciona as colunas necessárias nas bases
df_train_eval['base'] = 'treino'
df_test_eval['base'] = 'teste'
df_oot_eval['base'] = 'oot'

# Junta as bases
df_total = pd.concat([df_train_eval, df_test_eval, df_oot_eval], ignore_index=True)

# Calcula o KS por safra
ks_safra_df = calcular_ks_por_safra(df_total)

# Exibe
print("KS por safra:")
display(ks_safra_df)
import matplotlib.pyplot as plt

# Plotando o KS por safra
plt.figure(figsize=(10, 6))
plt.plot(ks_safra_df['Safra'], ks_safra_df['KS_Score'], marker='o', linestyle='-', color='blue')
plt.title('KS por Safra', fontsize=14)
plt.xlabel('Safra', fontsize=12)
plt.ylabel('KS Score', fontsize=12)
plt.grid(True)
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
%md
**Target por faixa, por safra**
#Aplicar quantis e calcular faixas na base completa

# Calcular os quantis com base no score do treino original
quantiles = np.quantile(y_pred_train, q=np.arange(0.1, 1.0, 0.1))

# Criar dataframe unificado com colunas necessárias
df_fx_train['origem'] = 'treino'
df_fx_test['origem'] = 'teste'
df_fx_oot['origem'] = 'oot'

df_final = pd.concat([df_fx_train, df_fx_test, df_fx_oot], axis=0)
df_final['faixa_score'] = df_final['score'].apply(lambda x: atribuir_faixa_score(x, quantiles))
#Gerar gráfico de performance (%target) por faixa e safra

# Agrupar por safra e faixa
df_plot = df_final.groupby(['Safra', 'faixa_score']).agg(
volume=('target', 'count'),
target=('target', 'sum')
).reset_index()

df_plot['percentual_target'] = (df_plot['target'] / df_plot['volume']) * 100

# Pivot para facilitar o gráfico
pivot_plot = df_plot.pivot(index='Safra', columns='faixa_score', values='percentual_target')

# Plotar gráfico
plt.figure(figsize=(16, 8))
for coluna in pivot_plot.columns:
plt.plot(pivot_plot.index, pivot_plot[coluna], marker='o', label=coluna)

plt.title('% Target por Faixa de Score ao Longo das Safras', fontsize=14)
plt.xlabel('Safra')
plt.ylabel('% Target')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

%md
**Volumetria por faixa, por safra, inclusive em safras recentes**
#Juntar Base Completa + Safras Recentes

# Assumindo que você tenha uma base de safra recente (sem target), tipo:
df_safras_recentes = pd.DataFrame({
# Supondo que tenha as mesmas colunas, especialmente "score" e "Safra"
})

# Se não tiver target, preenche com NaN para manter compatibilidade
df_safras_recentes['target'] = np.nan
df_safras_recentes['origem'] = 'safras_recentes'

# Juntar tudo
df_unificado = pd.concat([df_fx_train, df_fx_test, df_fx_oot, df_safras_recentes], axis=0)

# Aplicar a classificação em faixas de score usando quantis do treino
df_unificado['faixa_score'] = df_unificado['score'].apply(lambda x: atribuir_faixa_score(x, quantiles))
#Cálculo de Volumetria e %Target
df_resumo = df_unificado.groupby(['Safra', 'faixa_score']).agg(
volume=('score', 'count'),
target=('target', 'sum')
).reset_index()

df_resumo['percentual_volume'] = (df_resumo['volume'] / df_resumo.groupby('Safra')['volume'].transform('sum')) * 100
df_resumo['percentual_target'] = (df_resumo['target'] / df_resumo['volume']) * 100
#Gráficos

pivot_vol = df_resumo.pivot(index='Safra', columns='faixa_score', values='percentual_volume')

plt.figure(figsize=(18, 10))
for coluna in pivot_vol.columns:
plt.plot(pivot_vol.index, pivot_vol[coluna], marker='o', label=coluna)

plt.title('% Volumetria por Faixa de Score - Todas Safras', fontsize=16)
plt.xlabel('Safra')
plt.ylabel('% Volumetria')
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()
%md
# Registro MLFlow
mlflow.autolog(disable = True)
# Start an MLflow run
with mlflow.start_run(experiment_id="1753456477010896", run_name="modelo_bhv_con_pf_01_v1",
                      tags = {"MODELO":"LIGHTGBM",
                              "TRATATIVA_DADOS":"BALANCEADOS",
                              "AMOSTRAGEM":"ESTRATIFICADA_EVER90MOB12",
                              "TIPO_MODELO":"CLASSIFICATIVO"
                              }) as run:

    # Get the best hyperparameters
    best_hyperparameters = {'n_estimators': 100, 'learning_rate': 0.09778750519348421, 'num_leaves': 17, 'max_depth': 9, 'feature_fraction': 0.5024892755136359, 'feature_fraction_bynode': 0.6908844309006715, 'min_child_weight': 0.2584541291654643, 'max_delta_step': 4.58793077046451, 'min_data_in_leaf': 79, 'min_child_samples': 41, 'subsample': 0.5008940682011966, 'colsample_bytree': 0.7882288650456512, 'reg_alpha': 0.11697053640329573, 'reg_lambda': 2.389932286461989e-09, 'max_bin': 28}

    # Log the best hyperparameters
    mlflow.log_params(best_hyperparameters)

    # log selected inputs
    best_model_inputs = ['MEAN_VLRMULTJURPAGO','MEAN_VLRATRASO' 'vlr_restritivo', 'qtd_meses_tmp_prim_relac_bcp_bcp', 'qtd_idade_anos', 'MEAN_PERCTXADMPAGA', 'MEAN_PERCFDOAQUISICAO', 'qtd_dias_entrega_bem', 'flg_cotas_emdia', 'qtd_meses_restantes', 'MEAN_VLRLANCNORMPAGO', 'MEAN_vlr_diff_cart_cred', 'MEAN_VLRLANCEMBPAGO', 'flg_possui_aut', 'flg_possui_imv', 'flg_grupo_andamento', 'qtd_total_cotas', 'flg_norte_nordeste', 'vlr_parcela_sob_renda']

    mlflow.log_param("best_model_inputs",best_model_inputs)

    lgbm_best = final_model
    predictions = lgbm_best.predict_proba(X_train)[:, 1]
    signature = infer_signature(X_train, predictions)

    # Metrics
    mlflow.log_metric("KS Treino",ks_treino)
    mlflow.log_metric("KS Out of Time",ks_oot)
    mlflow.log_metric("Diff KS",ks_diff)

    # Log the best model
    mlflow.lightgbm.log_model(lgbm_best, "model",signature=signature)

    # Log KS plot
    mlflow.log_artifact(ks_summary_path)

    # Log feature importance plot
    mlflow.log_artifact(feature_importance_summary_path)

    # Log SHAP summary plot
    mlflow.log_artifact(shap_summary_path)

    # Taxa target por faixa
    mlflow.log_artifact("/dbfs/tmp/taxa_inadimplencia_bhv_con_pf_01_fx_train.csv")

    # Taxa target por faixa e por safra
    mlflow.log_artifact("/dbfs/tmp/taxa_inadimplencia_safras_bhv_con_pf_01_fx_train.csv")